[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_25_dropout_pure_solution.ipynb)

# 🟢 Solution: Dropout without Flax

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `b_25_dropout_pure.ipynb` first.

---
Problem 17's dropout, with no module to hold the RNG.

### Signature
```python
def apply_dropout(x, key, p, *, deterministic=False):
    ...   # -> same shape as x
```

There are no learnable parameters, so there is no `init_dropout`. What there
*is* — and what `nnx.Rngs` was quietly doing for you — is a key that has to
come from somewhere and be different on every call.

### Rules
- `deterministic=True` → return `x` unchanged.
- `p == 0.0` → return `x` unchanged, exactly. Both conditions, not just the first.
- Otherwise: keep each element with probability `1 - p`, and divide the
  survivors by `1 - p` so the expected value is unchanged (inverted dropout).

`p` and `deterministic` are hyperparameters, so under `jit` they are **static**
(`static_argnums=(2,)`, `static_argnames=('deterministic',)`) — the `p == 0`
exit is a Python branch and needs a concrete value.

### The key is the whole point
In problem 17 you wrote `self.rngs.dropout()` and a fresh mask appeared each
call, because the stream advanced itself. Here **nothing advances anything**:

```python
apply_dropout(x, key, 0.5)      # same key
apply_dropout(x, key, 0.5)      # SAME MASK — this is not a bug
```

Passing the same key twice gives the same mask, because JAX random functions
are pure. Two training steps need two keys, and that is the caller's job:

```python
key, sub = jax.random.split(key)
h = apply_dropout(h, sub, p)
```

That is what a module was hiding. It is also why a JAX training loop threads a
key through its carry — see `b_19`.

### Why divide by 1-p
So that `E[out] == x`. Doing it at training time ("inverted dropout") means
inference is a plain no-op instead of a rescale — which is exactly why
`deterministic=True` can just return `x`.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def apply_dropout(x, key, p, *, deterministic=False):
    # Both early exits: p == 0 must be an exact no-op, not "a mask that happens
    # to keep everything", which would still burn the key.
    if deterministic or p == 0.0:
        return x

    # bernoulli's p is the probability of True, i.e. the KEEP probability.
    keep = jax.random.bernoulli(key, 1.0 - p, x.shape)
    # Scale at train time so inference needs no rescale at all.
    return jnp.where(keep, x / (1.0 - p), 0.0)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

x = jnp.ones((8,))
key = jax.random.key(0)

print("p=0.5      ", apply_dropout(x, key, 0.5))
print("same key   ", apply_dropout(x, key, 0.5), "  <- identical, by design")
print("split key  ", apply_dropout(x, jax.random.split(key)[0], 0.5))
print("determinist", apply_dropout(x, key, 0.5, deterministic=True))

big = jnp.ones((100000,))
out = apply_dropout(big, key, 0.3)
print(f"\nmean over 100k: {float(jnp.mean(out)):.4f}  (should be ~1.0)")
print(f"fraction zeroed: {float(jnp.mean(out == 0)):.4f}  (should be ~0.3)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("dropout_pure")